In [1]:
# CELL 1 — load & clean
import pandas as pd
import os

spam_path = "spam.csv"   

if not os.path.exists(spam_path):
    raise FileNotFoundError(f"File not found: {spam_path}. Put it in the working directory or change spam_path.")

# The Kaggle SMS dataset has the label in col0 and message in col1; other cols are empty.
df = pd.read_csv(spam_path, encoding='latin-1', header=None, usecols=[0,1], names=['label','text'])

print("Loaded file:", spam_path)
print("Shape:", df.shape)
print("\nLabel counts:")
print(df['label'].value_counts())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nSample messages:")
display(df.head(10))

# Map labels to numeric for modeling
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Save cleaned version for later steps
os.makedirs("data", exist_ok=True)
clean_path = "data/spam_clean.csv"
df.to_csv(clean_path, index=False)
print("\nCleaned data saved ->", clean_path)


Loaded file: spam.csv
Shape: (5573, 2)

Label counts:
label
ham     4825
spam     747
v1         1
Name: count, dtype: int64

Missing values per column:
label    0
text     0
dtype: int64

Sample messages:


,label,text
0,v1,v2
1,ham,"Go until jurong point, crazy.. Available only ..."
2,ham,Ok lar... Joking wif u oni...
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...
4,ham,U dun say so early hor... U c already then say...
5,ham,"Nah I don't think he goes to usf, he lives aro..."
6,spam,FreeMsg Hey there darling it's been 3 week's n...
7,ham,Even my brother is not like to speak with me. ...
8,ham,As per your request 'Melle Melle (Oru Minnamin...
9,spam,WINNER!! As a valued network customer you have...



Cleaned data saved -> data/spam_clean.csv


In [2]:
# drop extra header row, preprocess, split, and vectorize
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Load cleaned file again
df = pd.read_csv("data/spam_clean.csv")

# Drop extra 'v1','v2' row if present
df = df[df['label'] != 'v1'].copy()
df.reset_index(drop=True, inplace=True)

# Simple text preprocessing: lowercase, remove non-alphanumeric chars
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['clean_text'] = df['text'].apply(clean_text)

# Convert labels to numeric
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num']
)

# TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)
print("\nSample cleaned text:")
print(X_train.iloc[0])


Train shape: (4457, 3000)
Test shape: (1115, 3000)

Sample cleaned text:
going on nothing greatbye


In [3]:
# CELL 3 — Train + Evaluate Logistic Regression model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train model
model = LogisticRegression(max_iter=2000)
model.fit(X_train_tfidf, y_train)

# Predictions
y_pred = model.predict(X_test_tfidf)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham','Spam']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.9713

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.99      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[965   1]
 [ 31 118]]


In [4]:
# CELL 4 — Train and evaluate XGBoost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_model.fit(X_train_tfidf, y_train, eval_set=[(X_test_tfidf, y_test)], verbose=False)

# Predict
y_pred_xgb = xgb_model.predict(X_test_tfidf)

# Evaluate
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Ham','Spam']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))


C:\Users\syeds\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [23:14:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Accuracy: 0.9739910313901345

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      0.99      0.99       966
        Spam       0.96      0.84      0.90       149

    accuracy                           0.97      1115
   macro avg       0.97      0.92      0.94      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[961   5]
 [ 24 125]]


In [5]:
import joblib
import os


os.makedirs("models", exist_ok=True)

# Save XGBoost model and TF-IDF vectorizer
joblib.dump(xgb_model, "models/spam_xgboost_model.joblib")
joblib.dump(tfidf, "models/spam_tfidf_vectorizer.joblib")

print("✅ Spam Defender model and vectorizer saved successfully!")


✅ Spam Defender model and vectorizer saved successfully!


In [6]:
# CELL A — save model + vectorizer
import joblib, os, json

os.makedirs("models", exist_ok=True)

# Save XGBoost model as joblib (.pkl) and json
joblib.dump(xgb_model, "models/spam_xgboost_model.pkl")   # PKL as you wanted
xgb_model.save_model("models/spam_xgboost_model.json")    # json copy (optional)

# Save TF-IDF vectorizer
joblib.dump(tfidf, "models/spam_tfidf_vectorizer.pkl")

# Save a small metadata JSON
meta = {
    "model": "spam_xgboost",
    "model_file_pkl": "models/spam_xgboost_model.pkl",
    "model_file_json": "models/spam_xgboost_model.json",
    "vectorizer": "models/spam_tfidf_vectorizer.pkl",
    "features": "tfidf(max_features=3000)",
    "label_map": {"ham":0,"spam":1}
}
with open("models/spam_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved: models/spam_xgboost_model.pkl, models/spam_xgboost_model.json, models/spam_tfidf_vectorizer.pkl, models/spam_meta.json")


Saved: models/spam_xgboost_model.pkl, models/spam_xgboost_model.json, models/spam_tfidf_vectorizer.pkl, models/spam_meta.json


In [7]:
# run defender (scores, actions, logs, blocklist)
import pandas as pd, joblib, os, numpy as np
from sklearn.metrics import accuracy_score, classification_report

# params: adjust thresholds here
BLOCK_THRESHOLD = 0.90   # >= -> block/quarantine
MONITOR_THRESHOLD = 0.60 # >= and < block -> monitor
# < monitor -> allow

# load artifacts (safe even if they are already in memory)
model = joblib.load("models/spam_xgboost_model.pkl")
vectorizer = joblib.load("models/spam_tfidf_vectorizer.pkl")


try:
    X_eval = X_test_tfidf
except NameError:
    # if X_test (raw text) is available:
    if 'X_test' in globals():
        X_eval = vectorizer.transform(X_test)
    else:
        raise NameError("X_test_tfidf not in memory. Rerun preprocessing or provide input to defend.")

# Probabilities and predictions
probs = model.predict_proba(X_eval)[:,1]   # prob of spam (class 1)
preds = (probs >= 0.5).astype(int)

# Build defender actions
actions = []
for p, lab in zip(probs, preds):
    if p >= BLOCK_THRESHOLD:
        actions.append("quarantine")     # or block_url/block_ip
    elif p >= MONITOR_THRESHOLD:
        actions.append("monitor")
    else:
        actions.append("allow")

# Build results DF
results = pd.DataFrame({
    "predicted_label": preds,
    "prediction_prob": probs,
    "defense_action": actions
})

# Add optional identifier: try to attach original index/text if available
if 'X_test' in globals():   # X_test was text series used to create tfidf
    results['text_sample'] = X_test.reset_index(drop=True)
else:
    results['text_sample'] = range(len(results))

# Save defender outputs
os.makedirs("models", exist_ok=True)
results.to_csv("models/spam_defense_results.csv", index=False)
results.to_csv("models/spam_defense_results_backup.csv", index=False)  # backup

# Action log (audit)
action_log = results.copy()
action_log['timestamp'] = pd.Timestamp.now()
action_log.to_csv("models/spam_defender_action_log.csv", index=False)

# Blocklist (only quarantined)
blocklist = action_log[action_log['defense_action'] == 'quarantine'][['text_sample','prediction_prob','timestamp']]
blocklist.to_csv("models/spam_blocklist.csv", index=False)

# Print quick stats
print("Defender results rows:", len(results))
print(results['defense_action'].value_counts())
print("Saved -> models/spam_defense_results.csv, models/spam_defender_action_log.csv, models/spam_blocklist.csv")


Defender results rows: 1115
defense_action
allow         990
quarantine    114
monitor        11
Name: count, dtype: int64
Saved -> models/spam_defense_results.csv, models/spam_defender_action_log.csv, models/spam_blocklist.csv


In [8]:
# feature importance & summary
import json, numpy as np, pandas as pd
from collections import OrderedDict

# load model & vectorizer
model = joblib.load("models/spam_xgboost_model.pkl")
vectorizer = joblib.load("models/spam_tfidf_vectorizer.pkl")

# get booster importance (gain)
booster = model.get_booster()
importance = booster.get_score(importance_type='gain')  # keys like 'f123'
# convert to sorted list
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)

# map f# -> actual token if possible
feature_names = vectorizer.get_feature_names_out()
mapped = []
for fname, val in importance_sorted[:40]:
    if fname.startswith('f'):
        idx = int(fname[1:])
        token = feature_names[idx] if idx < len(feature_names) else fname
    else:
        token = fname
    mapped.append((token, float(val)))

# Save top features
summary = {
    "top_features_gain": mapped[:50],
    "num_features": len(feature_names)
}
with open("models/spam_artifacts_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save CSV for quick viewing
pd.DataFrame(mapped, columns=["feature","gain"]).to_csv("models/spam_feature_importance.csv", index=False)

print("Saved feature importance and summary -> models/spam_artifacts_summary.json")
print("Top 10 features:", mapped[:10])


Saved feature importance and summary -> models/spam_artifacts_summary.json
Top 10 features: [('txt', 50.30438232421875), ('call', 39.978126525878906), ('information', 20.855804443359375), ('claim', 17.540790557861328), ('tone', 16.554401397705078), ('reply', 14.414716720581055), ('send', 14.05395221710205), ('unsubscribe', 12.778156280517578), ('text', 12.247489929199219), ('stop', 11.172635078430176)]


In [9]:
# backup artifacts
import shutil, os
os.makedirs("backup_models", exist_ok=True)
files = [
    "models/spam_xgboost_model.pkl",
    "models/spam_xgboost_model.json",
    "models/spam_tfidf_vectorizer.pkl",
    "models/spam_meta.json",
    "models/spam_defense_results.csv",
    "models/spam_defender_action_log.csv",
    "models/spam_blocklist.csv",
    "models/spam_feature_importance.csv",
    "models/spam_artifacts_summary.json"
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, "backup_models")
        print("Backed up:", f)
    else:
        print("Missing (not backed up):", f)
print("Backup complete -> backup_models/")


Backed up: models/spam_xgboost_model.pkl
Backed up: models/spam_xgboost_model.json
Backed up: models/spam_tfidf_vectorizer.pkl
Backed up: models/spam_meta.json
Backed up: models/spam_defense_results.csv
Backed up: models/spam_defender_action_log.csv
Backed up: models/spam_blocklist.csv
Backed up: models/spam_feature_importance.csv
Backed up: models/spam_artifacts_summary.json
Backup complete -> backup_models/


In [11]:
import os
import joblib
import pandas as pd
import numpy as np

# Try to import SHAP safely
try:
    import shap
    shap_available = True
except ModuleNotFoundError:
    print("SHAP not available, explainability plots will be skipped.")
    shap_available = False

# Load model and vectorizer
model = joblib.load("models/spam_xgboost_model.pkl")
vectorizer = joblib.load("models/spam_tfidf_vectorizer.pkl")

# Load some sample data
df = pd.read_csv("data/spam_clean.csv")  # or your test dataset
X = vectorizer.transform(df['text'])
y = df['label']

# Generate predictions
preds = model.predict(X)
probs = model.predict_proba(X)[:,1]

# Create explainability log only if SHAP is available
if shap_available:
    explainer = shap.Explainer(model)
    shap_values = explainer(X)
    
    # Example: save mean absolute SHAP values per feature
    feature_importance = pd.DataFrame({
        'feature': vectorizer.get_feature_names_out(),
        'shap_importance': np.abs(shap_values.values).mean(axis=0)
    }).sort_values(by='shap_importance', ascending=False)
    
    os.makedirs("models", exist_ok=True)
    feature_importance.to_csv("models/spam_explainability_log.csv", index=False)
    print("Explainability log saved -> models/spam_explainability_log.csv")
else:
    print("Skipped explainability log generation due to missing SHAP.")


SHAP not available, explainability plots will be skipped.
Skipped explainability log generation due to missing SHAP.


In [12]:
!pip install shap --quiet



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
import os

# ==========================
# Load Models and Data
# ==========================
model_path = "models/spam_xgboost_model.pkl"
vectorizer_path = "models/spam_tfidf_vectorizer.pkl"
defense_log_path = "models/spam_defense_results.csv"
explain_log_path = "models/spam_explain_log.csv"

model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)
defense_results = pd.read_csv(defense_log_path)

# ==========================
# Filter quarantined samples
# ==========================
quarantine_samples = defense_results[defense_results["defense_action"] == "quarantine"].copy()
print(f"Total quarantined samples: {len(quarantine_samples)}")

# ==========================
# Compute explainability (top TF-IDF features)
# ==========================
texts = quarantine_samples["text"].tolist()
X_tfidf = vectorizer.transform(texts)
feature_names = np.array(vectorizer.get_feature_names_out())

# Using SHAP for interpretability
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_tfidf)

# Prepare explainability results
explain_data = []
for i, msg in enumerate(texts):
    shap_row = shap_values[i].toarray().flatten()
    top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
    top_features = feature_names[top_indices]
    top_contribs = shap_row[top_indices]
    top_pairs = [f"{w} ({round(c, 3)})" for w, c in zip(top_features, top_contribs)]
    
    explain_data.append({
        "index": i,
        "text": msg[:120] + ("..." if len(msg) > 120 else ""),
        "top_contributing_words": ", ".join(top_pairs)
    })

# Convert to DataFrame
explain_df = pd.DataFrame(explain_data)

# Save explainability log
os.makedirs("models", exist_ok=True)
explain_df.to_csv(explain_log_path, index=False)
print(f"Explainability log saved -> {explain_log_path}")

explain_df.head()


ModuleNotFoundError: No module named 'shap'

In [2]:
!pip install shap

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
import os

# ==========================
# Load Models and Data
# ==========================
model_path = "models/spam_xgboost_model.pkl"
vectorizer_path = "models/spam_tfidf_vectorizer.pkl"
defense_log_path = "models/spam_defense_results.csv"
explain_log_path = "models/spam_explain_log.csv"

model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)
defense_results = pd.read_csv(defense_log_path)

# ==========================
# Filter quarantined samples
# ==========================
quarantine_samples = defense_results[defense_results["defense_action"] == "quarantine"].copy()
print(f"Total quarantined samples: {len(quarantine_samples)}")

# ==========================
# Compute explainability (top TF-IDF features)
# ==========================
texts = quarantine_samples["text"].tolist()
X_tfidf = vectorizer.transform(texts)
feature_names = np.array(vectorizer.get_feature_names_out())

# Using SHAP for interpretability
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_tfidf)

# Prepare explainability results
explain_data = []
for i, msg in enumerate(texts):
    shap_row = shap_values[i].toarray().flatten()
    top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
    top_features = feature_names[top_indices]
    top_contribs = shap_row[top_indices]
    top_pairs = [f"{w} ({round(c, 3)})" for w, c in zip(top_features, top_contribs)]
    
    explain_data.append({
        "index": i,
        "text": msg[:120] + ("..." if len(msg) > 120 else ""),
        "top_contributing_words": ", ".join(top_pairs)
    })

# Convert to DataFrame
explain_df = pd.DataFrame(explain_data)

# Save explainability log
os.makedirs("models", exist_ok=True)
explain_df.to_csv(explain_log_path, index=False)
print(f"Explainability log saved -> {explain_log_path}")

explain_df.head()


ModuleNotFoundError: No module named 'shap'

In [2]:
!{sys.executable} -m pip install shap


'{sys.executable}' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import sys
print(sys.executable)


C:\anaconda\python.exe


In [5]:
!python -m pip show shap


Name: shap
Version: 0.49.1
Summary: A unified approach to explain the output of any machine learning model.
Home-page: 
Author: 
Author-email: Scott Lundberg <slund1@cs.washington.edu>
License: MIT License
Location: c:\users\syeds\appdata\roaming\python\python39\site-packages
Requires: cloudpickle, numba, numpy, packaging, pandas, scikit-learn, scipy, slicer, tqdm, typing-extensions
Required-by: 


In [6]:
import sys
!C:\anaconda\python.exe -m pip show shap


In [7]:
!C:\anaconda\python.exe -m pip install shap


Defaulting to user installation because normal site-packages is not writeable
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
   ---------------------------------------- 0.0/548.0 kB ? eta -:--:--
   - ------------------------------------- 20.5/548.0 kB 682.7 kB/s eta 0:00:01
   -- ------------------------------------ 41.0/548.0 kB 495.5 kB/s eta 0:00:02
   ---- ---------------------------------- 61.4/548.0 kB 656.4 kB/s eta 0:00:01
   ---- ---------------------------------- 61.4/548.0 kB 656.4 kB/s eta 0:00:01
   ---- ---------------------------------- 61.4/548.0 kB 656.4 kB/s eta 0:00:01
   ---- ---------------------------------- 61.4/548.0 kB 656.4 kB/s eta 0:00:01
   ---- ---------------------------------- 61.4/548.0 kB 656.4 kB/s eta 0:00:01
   ---- ---------------------------------- 61.4/548.0 kB 656.4 kB/s eta 0:00:01
   ----- --------------------------------- 71.7/548.0 kB 163.8 kB/s eta 0:00:03
   ------ -------------------------------- 92.2/548.0 kB 187.5 kB/s 

In [6]:
import pandas as pd
import numpy as np
import joblib
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
import os

# ==========================
# Paths
# ==========================
model_path = "models/spam_xgboost_model.pkl"
vectorizer_path = "models/spam_tfidf_vectorizer.pkl"
defense_log_path = "models/spam_defense_results.csv"
explain_log_path = "models/spam_explain_log.csv"

# ==========================
# Load Models and Data
# ==========================
model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)
defense_results = pd.read_csv(defense_log_path)

# ==========================
# Identify quarantined samples
# ==========================
quarantine_samples = defense_results[defense_results["defense_action"] == "quarantine"].copy()
print(f"Total quarantined samples: {len(quarantine_samples)}")

# ==========================
# Ensure correct text column and convert to string
# ==========================
if "text" not in quarantine_samples.columns:
    # Rename if original column is different
    quarantine_samples.rename(columns={"text_sample": "text"}, inplace=True)

# Convert all text to string (fixes integer issues)
quarantine_samples["text"] = quarantine_samples["text"].astype(str)
texts = quarantine_samples["text"].tolist()

# ==========================
# TF-IDF transform
# ==========================
X_tfidf = vectorizer.transform(texts)
feature_names = np.array(vectorizer.get_feature_names_out())

# ==========================
# SHAP Explainer
# ==========================
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_tfidf)

# ==========================
# Prepare explainability results
# ==========================
explain_data = []
for i, msg in enumerate(texts):
    # shap_values[i] is numpy.ndarray
    shap_row = shap_values[i].flatten()
    top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
    top_features = feature_names[top_indices]
    top_contribs = shap_row[top_indices]
    top_pairs = [f"{w} ({round(c, 3)})" for w, c in zip(top_features, top_contribs)]
    
    explain_data.append({
        "index": quarantine_samples.index[i],
        "text": msg[:120] + ("..." if len(msg) > 120 else ""),
        "top_contributing_words": ", ".join(top_pairs)
    })

# ==========================
# Convert to DataFrame and save
# ==========================
explain_df = pd.DataFrame(explain_data)
os.makedirs("models", exist_ok=True)
explain_df.to_csv(explain_log_path, index=False)
print(f"Explainability log saved -> {explain_log_path}")

# Preview
explain_df.head(10)


Total quarantined samples: 114
Explainability log saved -> models/spam_explain_log.csv


,index,text,top_contributing_words
0,35,wanna get laid 2nite want real dogging locatio...,"txt (4.197000026702881), now (1.44200003147125..."
1,40,do you want 750 anytime any network mins 150 t...,"call (3.1110000610351562), reply (1.3380000591..."
2,65,urgent your mobile no was awarded a 2000 bonus...,"call (3.878000020980835), mobile (1.2330000400..."
3,67,todays voda numbers ending with 7634 are selec...,"call (3.2209999561309814), claim (2.3420000076..."
4,83,guess what somebody you know secretly fancies ...,"call (4.501999855041504), from (1.491000056266..."
5,89,sms auction a brand new nokia 7250 is up 4 au...,"txt (3.819000005722046), sms (2.02200007438659..."
6,94,you have won a guaranteed 1000 cash or a 2000 ...,"call (3.424999952316284), claim (1.87300002574..."
7,122,dear voucher holder 2 claim your 1st class air...,"call (3.513000011444092), claim (2.20600008964..."
8,123,talk sexy make new friends or fall in love in ...,"text (1.8259999752044678), service (1.21399998..."
9,127,win we have a winner mr t foley won an ipod mo...,"win (2.753999948501587), mobile (1.25399994850..."


In [7]:
import os
import joblib
import xgboost as xgb
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------
# 1️⃣ Set threat
# -------------------------------
THREAT_NAME = "spam"  

# -------------------------------
# 2️⃣ Paths
# -------------------------------
model_json = f"models/{THREAT_NAME}_xgb.json"
pkl_model = f"models/{THREAT_NAME}_xgboost_model.pkl"
defense_csv = f"models/{THREAT_NAME}_defense_results.csv"
vectorizer_path = f"models/{THREAT_NAME}_tfidf_vectorizer.pkl"
explain_log = f"models/{THREAT_NAME}_explain_log.csv"

# -------------------------------
# 3️⃣ Convert JSON → PKL if not exists
# -------------------------------
if not os.path.exists(pkl_model) and os.path.exists(model_json):
    booster = xgb.Booster()
    booster.load_model(model_json)
    clf = XGBClassifier()
    clf._Booster = booster
    clf._le = None
    joblib.dump(clf, pkl_model)
    print(f"Saved PKL model -> {pkl_model}")
else:
    print(f"PKL model exists -> {pkl_model}")

# -------------------------------
# 4️⃣ Load model and defense log
# -------------------------------
model = joblib.load(pkl_model)
defense_results = pd.read_csv(defense_csv)
print("Columns in defense CSV:", defense_results.columns.tolist())

# -------------------------------
# 5️⃣ Determine text column
# -------------------------------
text_column = None
for col in ["text", "text_sample"]:
    if col in defense_results.columns:
        text_column = col
        break

if text_column:
    defense_results.rename(columns={text_column: "text"}, inplace=True)
else:
    print("⚠️ No text column found. SHAP will use IDs instead.")

# -------------------------------
# 6️⃣ Filter quarantined samples
# -------------------------------
quarantine_samples = defense_results[defense_results["defense_action"] == "quarantine"].copy()
print(f"Total quarantined samples: {len(quarantine_samples)}")

# -------------------------------
# 7️⃣ TF-IDF Vectorizer
# -------------------------------
if os.path.exists(vectorizer_path):
    vectorizer = joblib.load(vectorizer_path)
else:
    vectorizer = TfidfVectorizer(max_features=3000)
    if "text" in defense_results.columns:
        vectorizer.fit(defense_results["text"].astype(str).tolist())
        joblib.dump(vectorizer, vectorizer_path)
        print(f"Saved new TF-IDF vectorizer -> {vectorizer_path}")
    else:
        vectorizer = None

if len(quarantine_samples) > 0 and vectorizer:
    X_features = vectorizer.transform(quarantine_samples["text"].astype(str).tolist())
    feature_names = np.array(vectorizer.get_feature_names_out())
else:
    X_features = None
    feature_names = None

# -------------------------------
# 8️⃣ SHAP Explainability
# -------------------------------
explain_data = []

if X_features is not None:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_features)

    for i in range(len(quarantine_samples)):
        shap_row = shap_values[i].flatten() if isinstance(shap_values, np.ndarray) else shap_values[i].toarray().flatten()
        top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
        top_feats = feature_names[top_indices]
        top_vals = shap_row[top_indices]
        top_pairs = [f"{f} ({round(v,3)})" for f,v in zip(top_feats, top_vals)]

        row_dict = {"index": quarantine_samples.index[i]}
        if "text" in quarantine_samples.columns:
            row_dict["text"] = quarantine_samples.iloc[i]["text"][:120] + ("..." if len(quarantine_samples.iloc[i]["text"]) > 120 else "")
        else:
            row_dict["sample_id"] = quarantine_samples.iloc[i].get("ip_or_id", quarantine_samples.index[i])

        row_dict["top_contributing_features"] = ", ".join(top_pairs)
        explain_data.append(row_dict)
else:
    print("⚠️ No quarantined samples or text data. Explain log will be empty.")

# -------------------------------
# 9️⃣ Save Explainability CSV
# -------------------------------
os.makedirs("models", exist_ok=True)
explain_df = pd.DataFrame(explain_data)
explain_df.to_csv(explain_log, index=False)
print(f"Explainability log saved -> {explain_log}")
print(f"✅ Completed: {THREAT_NAME}")


PKL model exists -> models/spam_xgboost_model.pkl
Columns in defense CSV: ['predicted_label', 'prediction_prob', 'defense_action', 'text_sample']
Total quarantined samples: 621
Explainability log saved -> models/spam_explain_log.csv
✅ Completed: spam
